## Phase 1: Predicting Drug resistance (y = Drugs, X = Mutations)

In [61]:
#Import packages
import pandas as pd

#Read in data
df = pd.read_csv("geno-pheno.dataset.tsv", sep = "\t")

df.head()

/tmp/ipykernel_7643/2676976629.py:5: DtypeWarning: Columns (0: AZT, 1: DDCFoldMatch, 2: TAFFoldMatch) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("geno-pheno.dataset.tsv", sep = "\t")


,RefID,IsolateID,IsolateName,Species,Type,PtID,Method,3TC,3TCFoldMatch,ABC,...,P291,P292,P293,P294,P295,P296,P297,P298,P299,P300
0,756,9918,CA9918,HIV1,Clinical,1391.0,PhenoSense,200.0,>,4.6,...,-,-,-,-,-,-,-,-,-,-
1,756,3832,CA3832,HIV1,Clinical,1433.0,PhenoSense,200.0,>,8.8,...,-,-,.,.,.,.,.,.,.,.
2,756,10464,CA10464,HIV1,Clinical,634.0,PhenoSense,200.0,>,14.0,...,.,.,.,.,.,.,.,.,.,.
3,756,9928,CA9928,HIV1,Clinical,637.0,PhenoSense,200.0,>,6.7,...,-,-,V,-,-,-,K,-,-,-
4,756,4372,CA4372,HIV1,Clinical,1274.0,PhenoSense,200.0,>,7.1,...,-,-,V,-,-,-,-,-,-,-


In [62]:
##Cleaning the data set
df_clean = df.copy()
df_clean.head()

,RefID,IsolateID,IsolateName,Species,Type,PtID,Method,3TC,3TCFoldMatch,ABC,...,P291,P292,P293,P294,P295,P296,P297,P298,P299,P300
0,756,9918,CA9918,HIV1,Clinical,1391.0,PhenoSense,200.0,>,4.6,...,-,-,-,-,-,-,-,-,-,-
1,756,3832,CA3832,HIV1,Clinical,1433.0,PhenoSense,200.0,>,8.8,...,-,-,.,.,.,.,.,.,.,.
2,756,10464,CA10464,HIV1,Clinical,634.0,PhenoSense,200.0,>,14.0,...,.,.,.,.,.,.,.,.,.,.
3,756,9928,CA9928,HIV1,Clinical,637.0,PhenoSense,200.0,>,6.7,...,-,-,V,-,-,-,K,-,-,-
4,756,4372,CA4372,HIV1,Clinical,1274.0,PhenoSense,200.0,>,7.1,...,-,-,V,-,-,-,-,-,-,-


In [37]:
# Dimensionality of data
df_clean.shape

(2505, 335)

### Exploratory data analysis

In [68]:
#DRop columns we don't need in the training
df2 = df.drop(columns = ["RefID","Species","Type", "Method", "NNRTIDRMs", 'CompleteMutationListAvailable', 'Author','NonDRMs', 'Author','RefYear', 'MedlineID', 'Title',  'PtID'])
df2.head(n = 300)

#Remove Protease positions

df2 = df2.loc[:, ~df2.columns.str.match(r"^P\d+$")]
df2.head()
df2.shape

(2505, 23)

In [70]:
# Explore the data
#df2.describe()

#df2.info()

df2.columns

df2.shape

(2505, 23)

In [69]:
# Find reverse transcriptase columns
[col for col in df2.columns if "RT" in col.upper()]

['NRTIDRMs']

In [71]:
#Check which columns retained
df2.columns

Index(['IsolateID', 'IsolateName', '3TC', '3TCFoldMatch', 'ABC',
       'ABCFoldMatch', 'AZT', 'AZTFoldMatch', 'D4T', 'D4TFoldMatch', 'DDC',
       'DDCFoldMatch', 'DDI', 'DDIFoldMatch', 'FTC', 'FTCFoldMatch', 'ISL',
       'ISLFoldMatch', 'TAF', 'TAFFoldMatch', 'TDF', 'TDFFoldMatch',
       'NRTIDRMs'],
      dtype='str')

In [72]:
#View data 
df2.head()

,IsolateID,IsolateName,3TC,3TCFoldMatch,ABC,ABCFoldMatch,AZT,AZTFoldMatch,D4T,D4TFoldMatch,...,DDIFoldMatch,FTC,FTCFoldMatch,ISL,ISLFoldMatch,TAF,TAFFoldMatch,TDF,TDFFoldMatch,NRTIDRMs
0,9918,CA9918,200.0,>,4.6,=,0.6,=,1.0,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M184V
1,3832,CA3832,200.0,>,8.8,=,3.2,=,1.9,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"M41L, L74LV, M184V, L210W, T215Y"
2,10464,CA10464,200.0,>,14.0,=,307,=,6.4,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y"
3,9928,CA9928,200.0,>,6.7,=,6.5,=,1.6,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"M41L, M184V, T215Y"
4,4372,CA4372,200.0,>,7.1,=,0.8,=,1.3,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"D67N, K70G, M184V, T215F, K219Q"


In [73]:
#Identifying unique mutations in the dataset

df2.dtypes

IsolateID         int64
IsolateName         str
3TC             float64
3TCFoldMatch        str
ABC             float64
ABCFoldMatch        str
AZT              object
AZTFoldMatch        str
D4T             float64
D4TFoldMatch        str
DDC             float64
DDCFoldMatch        str
DDI             float64
DDIFoldMatch        str
FTC             float64
FTCFoldMatch        str
ISL             float64
ISLFoldMatch        str
TAF             float64
TAFFoldMatch        str
TDF             float64
TDFFoldMatch        str
NRTIDRMs            str
dtype: object

In [74]:
# Number of duplicate IsolateIDs
print("Duplicate IsolateIDs:", df2["IsolateID"].duplicated().sum())

Duplicate IsolateIDs: 0


In [75]:
# Abstract drugs
missing = (df2.isnull().sum().to_frame(name="Missing"))

missing["Percent"] = round(missing["Missing"] / len(df_clean) * 100, 2)

missing.sort_values("Percent", ascending=False)

,Missing,Percent
ISLFoldMatch,2473,98.72
ISL,2473,98.72
TAFFoldMatch,2409,96.17
TAF,2409,96.17
DDCFoldMatch,2036,81.28
DDC,2003,79.96
FTCFoldMatch,1948,77.76
FTC,1948,77.76
NRTIDRMs,512,20.44
TDF,493,19.68


In [76]:
#Measure completeness of Drug data
drug_cols = ["3TC", "ABC", "AZT", "D4T", "DDC", "DDI", "FTC", "ISL", "TAF", "TDF"]

available = df2[drug_cols].notna().sum().sort_values(ascending=False)

print(available)

AZT    2381
D4T    2377
DDI    2377
3TC    2359
ABC    2231
TDF    2012
FTC     557
DDC     502
TAF      96
ISL      32
dtype: int64


In [77]:
#DRop columns we don't need in the training
df_clean = df2.drop(columns = ["3TCFoldMatch", "ABCFoldMatch", "AZTFoldMatch", "D4TFoldMatch", "DDCFoldMatch", "DDIFoldMatch", "FTCFoldMatch", "ISLFoldMatch", "TAFFoldMatch", "TDFFoldMatch"])
df_clean.head(n = 300)

#Drop DRugs with < 1000 complete cells
df_clean = df_clean.drop(columns = ["DDC", "FTC", "ISL", "TAF"])
df_clean.head(n=30)



,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs
0,9918,CA9918,200.0,4.6,0.6,1.0,1.6,NaN,M184V
1,3832,CA3832,200.0,8.8,3.2,1.9,2.1,NaN,"M41L, L74LV, M184V, L210W, T215Y"
2,10464,CA10464,200.0,14.0,307,6.4,2.7,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y"
3,9928,CA9928,200.0,6.7,6.5,1.6,1.4,NaN,"M41L, M184V, T215Y"
4,4372,CA4372,200.0,7.1,0.8,1.3,1.9,NaN,"D67N, K70G, M184V, T215F, K219Q"
5,4391,CA4391,200.0,5.9,1.9,1.3,1.7,NaN,"D67N, K70R, M184V, K219Q"
6,9912,CA9912,200.0,18.0,100,11.0,3.5,NaN,"E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K..."
7,9945,CA9945,200.0,29.0,726,10.0,4.1,NaN,"M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W"
8,9916,CA9916,200.0,7.7,7.9,2.3,1.8,NaN,"D67N, K70R, M184V, K219Q"
9,9944,CA9944,200.0,8.5,14,1.7,1.7,NaN,"M41L, M184V, L210W, T215Y"


In [48]:
#Data types for df_clean
print(df_clean.dtypes)

IsolateID        int64
IsolateName        str
3TC            float64
ABC            float64
AZT             object
D4T            float64
DDI            float64
TDF            float64
NRTIDRMs           str
dtype: object


In [49]:
#Save cleaned data 
df_clean.to_csv("geno-pheno_clean.tsv", sep="\t", index=False)
df_clean.head()

,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs
0,9918,CA9918,200.0,4.6,0.6,1.0,1.6,NaN,M184V
1,3832,CA3832,200.0,8.8,3.2,1.9,2.1,NaN,"M41L, L74LV, M184V, L210W, T215Y"
2,10464,CA10464,200.0,14.0,307,6.4,2.7,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y"
3,9928,CA9928,200.0,6.7,6.5,1.6,1.4,NaN,"M41L, M184V, T215Y"
4,4372,CA4372,200.0,7.1,0.8,1.3,1.9,NaN,"D67N, K70G, M184V, T215F, K219Q"


In [81]:
# Remove Isolates that lack NRTIDRMs
df_clean.dropna(subset = ["NRTIDRMs"], inplace= True)
df_clean

,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs
0,9918,CA9918,200.0,4.6,0.6,1.0,1.6,NaN,M184V
1,3832,CA3832,200.0,8.8,3.2,1.9,2.1,NaN,"M41L, L74LV, M184V, L210W, T215Y"
2,10464,CA10464,200.0,14.0,307,6.4,2.7,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y"
3,9928,CA9928,200.0,6.7,6.5,1.6,1.4,NaN,"M41L, M184V, T215Y"
4,4372,CA4372,200.0,7.1,0.8,1.3,1.9,NaN,"D67N, K70G, M184V, T215F, K219Q"
...,...,...,...,...,...,...,...,...,...
2499,42734,CA42734,10.0,8.6,500.0,6.4,1.7,6.3,"M41L, E44A, T69D, M184MV, L210W, T215Y, K219R"
2500,63964,CA63964,NaN,16.0,657.0,8.2,3.1,3.5,"M41L, E44A, D67N, T69D, V75M, M184V, L210W, T215F"
2501,48124,CA48124,200.0,15.0,500.0,4.3,3.0,5.8,"M41L, E44D, D67N, T69AD, M184MV, L210W, T215Y,..."
2502,761443,MK-8591A-053_ISL_2,NaN,NaN,NaN,NaN,NaN,NaN,"L74I, M184V"


In [82]:
#Count Number of Isolates - Number reduced after pruning isolates lacking NRTI-DRMs
df_clean["IsolateID"].nunique()

1993

### Feature Matrix Generation

In [83]:
#Features (Mutations) Matrix generation (Using )

df_clean["Mutation_List"] = (df_clean["NRTIDRMs"].str.split(",").apply(lambda x: [m.strip() for m in x]))
df_clean[["IsolateName","NRTIDRMs", "Mutation_List"]].head(10)


,IsolateName,NRTIDRMs,Mutation_List
0,CA9918,M184V,[M184V]
1,CA3832,"M41L, L74LV, M184V, L210W, T215Y","[M41L, L74LV, M184V, L210W, T215Y]"
2,CA10464,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y","[M41L, E44A, D67N, T69D, M184V, L210W, T215Y]"
3,CA9928,"M41L, M184V, T215Y","[M41L, M184V, T215Y]"
4,CA4372,"D67N, K70G, M184V, T215F, K219Q","[D67N, K70G, M184V, T215F, K219Q]"
5,CA4391,"D67N, K70R, M184V, K219Q","[D67N, K70R, M184V, K219Q]"
6,CA9912,"E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K...","[E40F, M41L, D67N, V75M, M184V, L210W, T215Y, ..."
7,CA9945,"M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W","[M41L, D67N, T69D, K70R, V75M, M184V, T215F, K..."
8,CA9916,"D67N, K70R, M184V, K219Q","[D67N, K70R, M184V, K219Q]"
9,CA9944,"M41L, M184V, L210W, T215Y","[M41L, M184V, L210W, T215Y]"


In [92]:
## Create the feature matrix (X) - One Hot encoding of the 'Mutation_List

from sklearn.preprocessing import MultiLabelBinarizer

#Create multilabelBinarizer object
mlb = MultiLabelBinarizer()

# One-Hot encode ["Mutation_List"]
X = mlb.fit_transform(df_clean["Mutation_List"])

X = pd.DataFrame(X, columns= mlb.classes_, index=df_clean["IsolateID"])

X.head(n=30)

,A62AV,A62V,D67DEG,D67DG,D67DH,D67DN,D67E,D67EK,D67G,D67GS,...,V75M,V75MT,V75S,V75T,V75VA,V75VI,V75VIM,V75VM,Y115F,Y115YF
IsolateID,,,,,,,,,,,,,,,,,,,,,
9918,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3832,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10464,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9928,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4372,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4391,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9912,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
9945,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
9916,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [85]:
# Explore Feature Matrix (X)
print(X.shape)

(1993, 165)


In [86]:
#Inspect feature name (Make sure they are Mutations)
X.columns.tolist()[:20]

['A62AV',
 'A62V',
 'D67DEG',
 'D67DG',
 'D67DH',
 'D67DN',
 'D67E',
 'D67EK',
 'D67G',
 'D67GS',
 'D67GV',
 'D67H',
 'D67HN',
 'D67N',
 'D67NH',
 'D67NS',
 'D67NT',
 'D67S',
 'D67~',
 'E40F']

### Target Matrix Generation

In [ ]:
df_target = pd.read_csv("geno-pheno_clean.tsv",sep = "\t")
df_target.head(n = 30)
df_target['Drug_List'] = df_target['Drugs'].apply(lambda s: [d.strip() for d in s.split(',') if d.strip()])

,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs
0,9918,CA9918,200.0,4.6,0.6,1.0,1.6,NaN,M184V
1,3832,CA3832,200.0,8.8,3.2,1.9,2.1,NaN,"M41L, L74LV, M184V, L210W, T215Y"
2,10464,CA10464,200.0,14.0,307,6.4,2.7,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y"
3,9928,CA9928,200.0,6.7,6.5,1.6,1.4,NaN,"M41L, M184V, T215Y"
4,4372,CA4372,200.0,7.1,0.8,1.3,1.9,NaN,"D67N, K70G, M184V, T215F, K219Q"
5,4391,CA4391,200.0,5.9,1.9,1.3,1.7,NaN,"D67N, K70R, M184V, K219Q"
6,9912,CA9912,200.0,18.0,100,11.0,3.5,NaN,"E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K..."
7,9945,CA9945,200.0,29.0,726,10.0,4.1,NaN,"M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W"
8,9916,CA9916,200.0,7.7,7.9,2.3,1.8,NaN,"D67N, K70R, M184V, K219Q"
9,9944,CA9944,200.0,8.5,14,1.7,1.7,NaN,"M41L, M184V, L210W, T215Y"


In [ ]:
# Create y - dataframe
df_target = df_clean[["IsolateID","3TC","ABC","AZT","D4T","DDI","TDF"]]
y.head(n = 30)

KeyError: 'drugs'

In [94]:
# Countercheck Isolate_Name compatibility between X and y

X = X.loc[y.index]

KeyError: "None of [Index([   0,    1,    2,    3,    4,    5,    6,    7,    8,    9,\n       ...\n       2494, 2495, 2496, 2497, 2498, 2499, 2500, 2501, 2502, 2503],\n      dtype='int64', length=1993)] are in the [index]"

## Phase 2: Predicting Drug resistance (y = Drugs, X = Mutations)

###### Data was retrieved from Stanford HIV Database (HIVDB) then transformed to long format using R 

In [ ]:
#Import geno-pheno_stan.tsv and append Isolate name 

gp_stan = pd.read_csv("~/Desktop/Eneza/eneza_project/HIV-drug-resistance-prediction-from-viral-sequences/Day2/genopheno_stan.tsv", sep = "\t")
gp_stan.head()

# Create a tsv file with appended isolate name
#gp_stan.to_csv("geno-pheno_clean.tsv", sep="\t", index=False)

# Quantifying missingness in data
missing = (gp_stan.isnull().sum().to_frame(name = "stan_Missing"))
missing["Percent"] = round(missing["stan_Missing"]/len(gp_stan)*100, 2)
missing.sort_values("Percent", ascending = False)


,stan_Missing,Percent
NRTIDRMs,3072,20.44
foldchange,1354,9.01
IsolateID,0,0.00
IsolateName,0,0.00
drugs,0,0.00


In [ ]:
# Remove Isolates that lack NRTIDRMs
gp_stan.dropna(subset = ["NRTIDRMs"], inplace= True)
gp_stan


,IsolateID,IsolateName,NRTIDRMs,drugs,foldchange
0,9918,CA9918,M184V,X3TC,200.0
1,9918,CA9918,M184V,ABC,4.6
2,9918,CA9918,M184V,AZT,0.6
3,9918,CA9918,M184V,D4T,1.0
4,9918,CA9918,M184V,DDI,1.6
...,...,...,...,...,...
15019,761442,MK-8591A-053_ISL_1,M184I,ABC,NaN
15020,761442,MK-8591A-053_ISL_1,M184I,AZT,NaN
15021,761442,MK-8591A-053_ISL_1,M184I,D4T,NaN
15022,761442,MK-8591A-053_ISL_1,M184I,DDI,NaN


In [ ]:
# Check completeness of data
available = gp_stan.notna().sum().sort_values(ascending=False)

print(available)


IsolateID      11958
IsolateName    11958
NRTIDRMs       11958
drugs          11958
foldchange     10693
dtype: int64


In [ ]:
# Check Data Types
gp_stan.dtypes

IsolateID        int64
IsolateName        str
NRTIDRMs           str
drugs              str
foldchange     float64
dtype: object

In [ ]:
#Count Number of Isolates - Number reduced after pruning isolates lacking NRTI-DRMs
gp_stan["IsolateID"].nunique()

1993

### Feature (X) and Target (y) Matrix Generation 

In [ ]:
#Features (Mutations) Matrix generation (Using )

gp_stan["Mutation_List"] = (gp_stan["NRTIDRMs"].str.split(",").apply(lambda x: [m.strip() for m in x]))
gp_stan[["IsolateName","NRTIDRMs", "Mutation_List"]].head(10)


,IsolateName,NRTIDRMs,Mutation_List
0,CA9918,M184V,[M184V]
1,CA9918,M184V,[M184V]
2,CA9918,M184V,[M184V]
3,CA9918,M184V,[M184V]
4,CA9918,M184V,[M184V]
5,CA9918,M184V,[M184V]
6,CA3832,"M41L, L74LV, M184V, L210W, T215Y","[M41L, L74LV, M184V, L210W, T215Y]"
7,CA3832,"M41L, L74LV, M184V, L210W, T215Y","[M41L, L74LV, M184V, L210W, T215Y]"
8,CA3832,"M41L, L74LV, M184V, L210W, T215Y","[M41L, L74LV, M184V, L210W, T215Y]"
9,CA3832,"M41L, L74LV, M184V, L210W, T215Y","[M41L, L74LV, M184V, L210W, T215Y]"


In [ ]:
# Create Drug List